# 03 · Gold models
Business-ready tables rebuilt from Silver: occupancy, rent roll, lease expiry, delinquency,
maintenance, collection trend, property performance and the portfolio KPI summary.

In [ ]:
%pip install -q pyyaml python-dateutil

In [ ]:
# Setup: make the repo's `src` package importable and select the Databricks environment.
# The notebook runs from <repo>/notebooks inside a Databricks Git folder.
import os, sys
repo_root = os.path.abspath("..")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
os.environ["LAKEHOUSE_ENV"] = "databricks"

dbutils.widgets.text("catalog", "workspace")          # Free Edition default; production: real_estate
catalog = dbutils.widgets.get("catalog")
overrides = {"environments": {"databricks": {"catalog": catalog,
                                             "raw_volume_root": f"/Volumes/{catalog}/raw/landing"}}}

from src.pipeline import build_lakehouse
lh, cfg = build_lakehouse(env="databricks", overrides=overrides)
print(f"catalog={cfg.catalog}  as_of={cfg.reporting_as_of}  batches={list(cfg.batches)}")

In [ ]:
from src.pipeline import run_gold
run_gold(lh, cfg)    # also runs the Gold data-quality suite
display(spark.table(f"{catalog}.gold.portfolio_summary"))

In [ ]:
display(spark.table(f"{catalog}.gold.property_performance").orderBy("occupancy_rate"))

In [ ]:
display(spark.table(f"{catalog}.gold.collection_trend"))

## Export the dashboard data (writes dashboard/data/dashboard_data.json in the Git folder)

In [ ]:
from src.dashboard.export_dashboard import export
print(export(lh, cfg))